# AutoPipe — CIFAR-10 con anomalías sintéticas (*one-vs-rest*)

Un protocolo clásico para convertir un dataset de clasificación en uno de
detección de anomalías es **one-vs-rest**: se elige una clase como *normal* y
se tratan muestras de las demás clases como *anomalías*.

Este ejemplo lo aplica sobre [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html)
y muestra el motor **Optuna** sobre un **espacio de búsqueda completo con las
7 transformaciones predefinidas** de la librería (incluidas las restricciones
de orden entre ellas). Si no hay CIFAR-10 disponible, usa un *fallback*
sintético para ejecutarse de extremo a extremo.

> **Datos reales:** pon `USE_REAL_CIFAR = True` (requiere `torchvision`); se
> descargará CIFAR-10 (~170 MB) automáticamente en `datasets/`.

In [1]:
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image
from sklearn.ensemble import IsolationForest

from autopipe import (
    CLAHETransform,
    GammaCorrection,
    GaussianBlur,
    GrayscaleConversion,
    ImageDataLoader,
    MedianFilter,
    Normalize,
    OptunaSearchEngine,
    ReportGenerator,
    Resize,
)

C:\Users\ricar\OneDrive - Estudiantes ITCR\2026 - I Semestre\Proyecto de Aplicación de Ingeniería\AutoPipe\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración

In [2]:
USE_REAL_CIFAR = False   # True para descargar CIFAR-10 vía torchvision
NORMAL_CLASS = 1         # clase considerada 'normal' (0-9)
N_NORMAL = 200           # muestras normales
N_ANOMALY = 40           # muestras anómalas (de otras clases)

## 2. Obtener imágenes y etiquetas binarias

`load_cifar10()` intenta cargar CIFAR-10 real; `synthesize_cifar()` genera un
sustituto separable (la clase normal es verdosa y de baja varianza; las
anomalías son rojizas y ruidosas).

In [3]:
def load_cifar10():
    """Carga CIFAR-10 (train) vía torchvision; devuelve (images, targets)."""
    from torchvision.datasets import CIFAR10
    ds = CIFAR10(root='datasets', train=True, download=True)
    images = np.stack([np.asarray(img) for img, _ in ds])
    targets = np.asarray(ds.targets)
    return images, targets


def synthesize_cifar(n_per_class: int = 60, n_classes: int = 10):
    """Genera imágenes 32x32 RGB sintéticas separables por clase."""
    rng = np.random.default_rng(0)
    images, targets = [], []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            if cls == NORMAL_CLASS:
                arr = np.clip(rng.normal([60, 150, 60], 12, (32, 32, 3)), 0, 255)
            else:
                arr = np.clip(rng.normal([170, 60, 60], 40, (32, 32, 3)), 0, 255)
            images.append(arr.astype(np.uint8))
            targets.append(cls)
    return np.stack(images), np.asarray(targets)

In [4]:
if USE_REAL_CIFAR:
    cifar_images, cifar_targets = load_cifar10()
    print('CIFAR-10 real cargado:', cifar_images.shape)
else:
    cifar_images, cifar_targets = synthesize_cifar()
    print('Usando CIFAR-10 sintético:', cifar_images.shape)

rng = np.random.default_rng(42)
normal_idx = rng.permutation(np.where(cifar_targets == NORMAL_CLASS)[0])[:N_NORMAL]
anomaly_idx = rng.permutation(np.where(cifar_targets != NORMAL_CLASS)[0])[:N_ANOMALY]
print('Normales:', len(normal_idx), '| anómalas:', len(anomaly_idx))

Usando CIFAR-10 sintético: (600, 32, 32, 3)
Normales: 60 | anómalas: 40


## 3. Escribir la estructura `normal/` + `anomaly/` y cargarla

In [5]:
dataset_dir = Path(tempfile.mkdtemp()) / 'cifar_ovr'
for folder, idxs in (('normal', normal_idx), ('anomaly', anomaly_idx)):
    target = dataset_dir / folder
    target.mkdir(parents=True, exist_ok=True)
    for i, idx in enumerate(idxs):
        Image.fromarray(cifar_images[idx]).save(target / f'img_{i:04d}.png')

images, labels = ImageDataLoader(dataset_path=dataset_dir).load()
print('Imágenes:', images.shape, '| etiquetas:', labels.shape)

Imágenes: (100, 32, 32, 3) | etiquetas: (100,)


## 4. Espacio de búsqueda completo (las 7 transformaciones predefinidas)

Este ejemplo usa **todas** las transformaciones de la librería: `Resize`,
`MedianFilter`, `GaussianBlur`, `CLAHETransform`, `GammaCorrection`,
`GrayscaleConversion` y `Normalize`. Además, el motor prueba automáticamente
**omitir** cada paso, así que Optuna también explora subconjuntos de esta cadena.

**Restricciones de orden a tener en cuenta:**

- `Normalize` debe ir **al final**: produce valores flotantes, mientras que
  `CLAHETransform` y la tabla de búsqueda de `GammaCorrection` operan sobre
  enteros (`uint8`). Si colocas `Normalize` antes de `CLAHETransform`, el
  `Pipeline` emite una advertencia de incompatibilidad por `logging`.
- `CLAHETransform` funciona tanto en color (canal L del espacio LAB) como en
  escala de grises, por lo que puede ir antes o después de
  `GrayscaleConversion`.
- Los filtros de ruido (`MedianFilter`, `GaussianBlur`) conviene aplicarlos
  antes de las mejoras de contraste, para no amplificar ruido.

In [6]:
# Las claves definen el ORDEN de aplicación; Normalize va al final porque
# produce flotantes y CLAHE/GammaCorrection esperan enteros (uint8).
search_space = {
    'resize':    [Resize(width=16, height=16)],
    'median':    [MedianFilter(kernel_size=3)],
    'blur':      [GaussianBlur(kernel_size=3), GaussianBlur(kernel_size=5)],
    'clahe':     [CLAHETransform(clip_limit=2.0, tile_grid_size=(4, 4))],
    'gamma':     [GammaCorrection(gamma=0.5), GammaCorrection(gamma=1.5)],
    'gray':      [GrayscaleConversion()],
    'normalize': [Normalize(mean=0.5, std=0.5)],
}

## 5. Búsqueda bayesiana con Optuna

In [7]:
# Con la opción automática de omitir cada paso, el espacio completo tiene
# cientos de combinaciones; Optuna lo explora con un presupuesto acotado.
engine = OptunaSearchEngine(search_space, n_trials=12, val_size=0.3, random_state=42)
best_pipeline, metrics, results = engine.run(
    images, labels, base_model=IsolationForest(random_state=0), metric='auc_roc',
)
print('Mejor pipeline:', best_pipeline)
for name, value in metrics.items():
    print(f'  {name:>20}: {value:.4f}')

[I 2026-07-01 16:06:26,186] A new study created in memory with name: no-name-c40e66b3-e005-4231-b4d7-28e06b4dfc0e


[I 2026-07-01 16:06:26,392] Trial 0 finished with value: 1.0 and parameters: {'resize': 1, 'median': 0, 'blur': 0, 'clahe': 0, 'gamma': 2, 'gray': 0, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:26,496] Trial 1 finished with value: 1.0 and parameters: {'resize': 1, 'median': 0, 'blur': 0, 'clahe': 1, 'gamma': 0, 'gray': 0, 'normalize': 0}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:26,603] Trial 2 finished with value: 1.0 and parameters: {'resize': 1, 'median': 0, 'blur': 2, 'clahe': 0, 'gamma': 2, 'gray': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:26,708] Trial 3 finished with value: 1.0 and parameters: {'resize': 0, 'median': 0, 'blur': 0, 'clahe': 0, 'gamma': 2, 'gray': 0, 'normalize': 0}. Best is trial 0 with value: 1.0.


...

[I 2026-07-01 16:06:27,223] Trial 8 finished with value: 1.0 and parameters: {'resize': 1, 'median': 0, 'blur': 2, 'clahe': 1, 'gamma': 2, 'gray': 0, 'normalize': 0}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:27,329] Trial 9 finished with value: 1.0 and parameters: {'resize': 0, 'median': 0, 'blur': 2, 'clahe': 0, 'gamma': 1, 'gray': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:27,437] Trial 10 finished with value: 1.0 and parameters: {'resize': 1, 'median': 1, 'blur': 0, 'clahe': 0, 'gamma': 1, 'gray': 1, 'normalize': 1}. Best is trial 0 with value: 1.0.


[I 2026-07-01 16:06:27,542] Trial 11 finished with value: 1.0 and parameters: {'resize': 1, 'median': 0, 'blur': 0, 'clahe': 1, 'gamma': 0, 'gray': 0, 'normalize': 1}. Best is trial 0 with value: 1.0.


Mejor pipeline: Pipeline([MedianFilter(kernel_size=3), GaussianBlur(kernel_size=3), CLAHETransform(clip_limit=2.0, tile_grid_size=(4, 4)), GrayscaleConversion()])
               auc_roc: 1.0000
                    f1: 1.0000
             precision: 1.0000
                recall: 1.0000
   false_positive_rate: 0.0000


## 6. Reporte

In [8]:
features = np.vstack([best_pipeline.apply(img).ravel() for img in images])
scorer = IsolationForest(random_state=0).fit(features)
scores = -scorer.score_samples(features)

# Sin output_dir la librería escribiría en reports/ junto al notebook; aquí
# usamos una subcarpeta explícita para no chocar con los demás ejemplos.
report = ReportGenerator(output_dir='reports/cifar', top_anomalies=6).generate(
    results, best_pipeline, metrics, metric='auc_roc', images=images, scores=scores,
)
print('HTML:', report['html'])
print('Config:', report['config'])

HTML: reports\cifar\report.html
Config: reports\cifar\winning_pipeline.json


## 7. Escalar a datos reales

Para usar CIFAR-10 real: pon `USE_REAL_CIFAR = True`, aumenta `N_NORMAL`/
`N_ANOMALY` y amplía el espacio de búsqueda o el número de `n_trials`. El resto
del flujo no cambia.